# XGBoost Training — Plegma Dataset

Trains one XGBoost binary classifier per appliance on the Plegma per-house Parquet files.

**Strategy:**
- Only houses that **have** a given appliance (and have ≥ 1 ON event) are included
- **House-based 80/20 train/test split** — whole houses go to train or test, never split row-wise
- **GroupKFold cross-validation** — CV folds respect house boundaries
- `scale_pos_weight` caps at 4× to handle class imbalance without over-correcting
- Best hyperparameters selected by grid search on macro F1

## Configuration

In [1]:
import os
import glob
import pickle
import warnings
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score,
                             roc_auc_score, classification_report)

warnings.filterwarnings("ignore")

PLEGMA_DIR  = r"C:\Users\moham\Documents\490 project new\plegma_houses"
MODELS_DIR  = r"C:\Users\moham\Documents\490 project new\final_models"
RANDOM_SEED = 42
MIN_HOUSES  = 3  # minimum houses needed to train a model

# Commented appliances were trained from another dataset

WANTED_APPLIANCES = [
    'elec_cooling_on',
    #'elec_heating_on',
    'elec_hot_water_on',
    #'elec_television_on',
    #'elec_range_oven_on',
    'elec_clothes_washer_on',
]

FEATURE_COLS = [
    'weather_drybulb_temp_c',
    'weather_relative_humidity_pct',
    'hour',
    'day_of_week',
    'is_weekend',
    'month',
]

PARAM_GRID = {
    "n_estimators":  [100, 300],
    "max_depth":     [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.3],
}

## Load All House Files

In [2]:
os.makedirs(MODELS_DIR, exist_ok=True)

house_files = sorted(glob.glob(os.path.join(PLEGMA_DIR, '*.parquet')))
if not house_files:
    raise FileNotFoundError(f"No parquet files found in {PLEGMA_DIR}")

print(f"Found {len(house_files)} house files\n")

# Load each house once and store in a dict
houses = {}
for f in house_files:
    df_h     = pd.read_parquet(f)
    house_id = df_h['house_id'].iloc[0]
    houses[house_id] = df_h
    print(f"  Loaded {house_id}: {len(df_h):,} rows | "
          f"appliances: {[c for c in df_h.columns if c in WANTED_APPLIANCES]}")

Found 13 house files

  Loaded House_01: 10,608 rows | appliances: ['elec_cooling_on', 'elec_hot_water_on', 'elec_clothes_washer_on']
  Loaded House_02: 4,578 rows | appliances: ['elec_cooling_on', 'elec_hot_water_on', 'elec_clothes_washer_on']
  Loaded House_03: 10,716 rows | appliances: ['elec_cooling_on', 'elec_hot_water_on', 'elec_clothes_washer_on']
  Loaded House_04: 8,521 rows | appliances: ['elec_cooling_on', 'elec_hot_water_on', 'elec_clothes_washer_on']
  Loaded House_05: 6,545 rows | appliances: ['elec_cooling_on', 'elec_hot_water_on', 'elec_clothes_washer_on']
  Loaded House_06: 4,369 rows | appliances: ['elec_hot_water_on', 'elec_clothes_washer_on']
  Loaded House_07: 8,008 rows | appliances: ['elec_cooling_on', 'elec_hot_water_on', 'elec_clothes_washer_on']
  Loaded House_08: 2,197 rows | appliances: ['elec_cooling_on', 'elec_clothes_washer_on']
  Loaded House_09: 5,803 rows | appliances: ['elec_hot_water_on', 'elec_clothes_washer_on']
  Loaded House_10: 3,900 rows | appl

## Training Loop — One Model per Appliance

### Eligible house selection & data combination

In [3]:
results = []

for target in WANTED_APPLIANCES:
    print(f"\n{'='*55}")
    print(f"APPLIANCE: {target}")
    print(f"{'='*55}")

    # Find houses that have this appliance column AND have ON events
    eligible_houses = {}
    for house_id, df_h in houses.items():
        if target not in df_h.columns:
            continue
        on_count = (df_h[target] == 1).sum()
        if on_count == 0:
            continue
        eligible_houses[house_id] = df_h

    n_eligible = len(eligible_houses)
    print(f"  Houses with this appliance: {n_eligible} "
          f"({list(eligible_houses.keys())})")

    if n_eligible < MIN_HOUSES:
        print(f"  [SKIPPED] Need at least {MIN_HOUSES} houses, only have {n_eligible}")
        continue

    # Combine only eligible houses
    df = pd.concat(
        [df_h[[*FEATURE_COLS, target, 'house_id']]
         for df_h in eligible_houses.values()],
        ignore_index=True
    )

    # Drop any remaining NaN rows
    before = len(df)
    df     = df.dropna()
    if before != len(df):
        print(f"  Dropped {before - len(df):,} NaN rows")

    print(f"  Combined rows: {len(df):,}")

    # ── House-based train/test split ──────────────────────────────────────────
    house_ids = np.array(list(eligible_houses.keys()))
    np.random.seed(RANDOM_SEED)
    np.random.shuffle(house_ids)

    split        = max(1, int(len(house_ids) * 0.8))
    train_houses = set(house_ids[:split])
    test_houses  = set(house_ids[split:])

    train_mask = df['house_id'].isin(train_houses)
    test_mask  = df['house_id'].isin(test_houses)

    X_train = df.loc[train_mask, FEATURE_COLS].reset_index(drop=True)
    X_test  = df.loc[test_mask,  FEATURE_COLS].reset_index(drop=True)
    y_train = df.loc[train_mask, target].astype(int).reset_index(drop=True)
    y_test  = df.loc[test_mask,  target].astype(int).reset_index(drop=True)
    groups  = df.loc[train_mask, 'house_id'].reset_index(drop=True)

    on_train = y_train.sum()
    on_test  = y_test.sum()
    print(f"  Train houses: {sorted(train_houses)} | rows: {len(X_train):,} | ON: {on_train:,}")
    print(f"  Test  houses: {sorted(test_houses)}  | rows: {len(X_test):,}  | ON: {on_test:,}")

    if on_train == 0:
        print(f"  [SKIPPED] No ON events in training set")
        continue

    # ── Train ─────────────────────────────────────────────────────────────────
    neg = (y_train == 0).sum()
    pos = on_train
    scale_pos_weight = min(neg / pos, 4.0)
    print(f"  scale_pos_weight: {scale_pos_weight:.2f}")

    base_model = XGBClassifier(
         scale_pos_weight=scale_pos_weight,
        eval_metric="aucpr",
        random_state=RANDOM_SEED,
        device="cuda",
        n_jobs=-1,
    )

    n_splits    = min(5, len(train_houses))
    group_kfold = GroupKFold(n_splits=n_splits)
    cv_splits   = group_kfold.split(X_train, y_train, groups=groups)

    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=PARAM_GRID,
        cv=cv_splits,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=0,
    )

    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_

    # ── Evaluate ──────────────────────────────────────────────────────────────
    y_pred  = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_test, y_proba)
    except ValueError:
        auc = float("nan")

    print(f"\n  Best params: {grid_search.best_params_}")
    print(f"  CV F1:       {grid_search.best_score_:.4f}")
    print(f"  Test Acc:    {acc:.4f} | Test F1: {f1:.4f} | AUC: {auc:.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))

    results.append({
        "appliance":  target,
        "houses":     n_eligible,
        "cv_f1":      round(grid_search.best_score_, 4),
        "test_f1":    round(f1,  4),
        "test_auc":   round(auc, 4),
        "test_acc":   round(acc, 4),
        "best_params": grid_search.best_params_,
    })

    model_path = os.path.join(MODELS_DIR, f"{target}.pkl")
    with open(model_path, "wb") as f:
        pickle.dump(best_model, f)
    print(f"  Saved → {model_path}")


APPLIANCE: elec_cooling_on
  Houses with this appliance: 11 (['House_01', 'House_02', 'House_03', 'House_04', 'House_05', 'House_07', 'House_08', 'House_10', 'House_11', 'House_12', 'House_13'])
  Combined rows: 70,617
  Train houses: [np.str_('House_01'), np.str_('House_02'), np.str_('House_03'), np.str_('House_05'), np.str_('House_07'), np.str_('House_11'), np.str_('House_12'), np.str_('House_13')] | rows: 55,999 | ON: 2,783
  Test  houses: [np.str_('House_04'), np.str_('House_08'), np.str_('House_10')]  | rows: 14,618  | ON: 762
  scale_pos_weight: 4.00

  Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
  CV F1:       0.5733
  Test Acc:    0.8985 | Test F1: 0.2352 | AUC: 0.8077
              precision    recall  f1-score   support

           0       0.96      0.93      0.95     13856
           1       0.19      0.30      0.24       762

    accuracy                           0.90     14618
   macro avg       0.58      0.62      0.59     14618
weighted avg

In [4]:
# ── 4. Summary ────────────────────────────────────────────────────────────────
if results:
    print(f"\n{'='*70}")
    print("SUMMARY")
    print(f"{'='*70}")
    summary = pd.DataFrame(results).drop(columns=["best_params"])
    print(summary.to_string(index=False))
else:
    print("\nNo models were trained.")


SUMMARY
             appliance  houses  cv_f1  test_f1  test_auc  test_acc
       elec_cooling_on      11 0.5733   0.2352    0.8077    0.8985
     elec_hot_water_on      12 0.5238   0.0771    0.6608    0.9303
elec_clothes_washer_on      13 0.5141   0.1261    0.6859    0.9227
